# Advanced Transformer-Based rPPG Pipeline — v2 (Fixed + Improved)

## What Was Fixed
| # | Bug | Impact | Fix |
|---|-----|--------|-----|
| 1 | `rgb_signals` accumulated **3× too many entries** (one per ROI, not per frame) | Corrupted POS signal frequency; BPM completely wrong | Average ROIs per frame → 1 entry per frame |
| 2 | Frame–signal **length mismatch** after `temporal_difference` (N→N-1) | Training on misaligned targets | Trim signal to `len(frames)` *after* differencing |
| 3 | **Global `face_mesh`** object — not picklable across DataLoader workers | Crash with `num_workers > 0` | Instantiate inside function via `worker_init_fn` & global dict |
| 4 | **Variable sequence length** — no padding/trimming to `SEQ_LEN` | `collate_fn` crash stacking tensors of different shapes | Sliding-window crop + zero-pad short videos |
| 5 | **No empty-detection guard** — `rgb_signals = []` → POS crash | Silent crash on videos with no face | Guard + skip frame; raise error if < min_frames |
| 6 | **No positional encoding** in Transformer | Model cannot distinguish temporal order | Add sinusoidal positional encoding |
| 7 | **Frame-by-frame CNN loop** in `forward()` | OOM / very slow on long sequences | Reshape to `(B*T, C, H, W)`, one batched forward |
| 8 | `dim_feedforward` default = 2048 for `d_model=128` | 16× oversized FFN; wasted params | Set `dim_feedforward=512` |
| 9 | `video_paths` points to **directory**, not video file | Silent failure — 0 frames loaded | Fixed placeholder + clear error message |
| 10 | No **validation split or metrics** | No way to monitor generalisation | Added val loop + MAE/BPM metrics |

## Improvements Implemented (from Future Improvements Section)
- **Attention Pooling** — replaces `AdaptiveAvgPool2d`; spatially weighted feature aggregation
- **Multi-Scale CNN Encoder** — extracts features at multiple spatial resolutions (TS-CAN inspired)
- **Dual-Stream (Appearance + Motion)** — separate CNN branches for raw frames and temporal differences
- **Skin Segmentation** — YCrCb-based skin mask applied before ROI to suppress non-skin pixels
- **Motion Augmentation** — random temporal jitter, horizontal flip, brightness/contrast perturbation
- **Enhanced Frequency-Domain Supervision** — BPM-band-masked spectral loss for sharper frequency alignment
- **Gradient Checkpointing** — reduces memory footprint for long sequences
- **Self-Supervised Pre-training Hook** — `ContrastiveRPPGHead` stub for future SSL training


In [ ]:
# Install dependencies (run once)
# !pip install torch torchvision torchaudio
# !pip install mediapipe opencv-python scipy numpy matplotlib tqdm

In [1]:
import os
import cv2
import math
import torch
import random
import mediapipe as mp
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from scipy.signal import butter, filtfilt
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint

2026-05-10 13:25:27.010441: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-10 13:25:27.012291: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-10 13:25:27.053768: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-10 13:25:28.025915: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Configuration

In [2]:
class CFG:
    DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
    IMG_SIZE   = 128
    SEQ_LEN    = 300        # fixed temporal window (frames)
    BATCH_SIZE = 2
    EPOCHS     = 100
    LR         = 1e-4
    D_MODEL    = 128
    NHEAD      = 8
    NUM_LAYERS = 4
    DIM_FF     = 512        # FIX #8: was defaulting to 2048 (way too large for d_model=128)
    DROPOUT    = 0.1
    FPS        = 30
    MIN_FRAMES = 64         # minimum valid frames in a video clip
    NUM_WORKERS = 0         # keep 0 unless you configure worker_init_fn (see DataLoader cell)
    GRAD_CKPT  = True       # enable gradient checkpointing for memory savings

cfg = CFG()
print("Device:", cfg.DEVICE)

Device: cpu


## FaceMesh — Thread-Safe Initialization

**FIX #3:** The original code created `face_mesh` as a **global singleton**, which cannot be
pickled across DataLoader worker processes (`num_workers > 0` crashes). We now create one
instance **per worker** using a worker-local dictionary keyed by `os.getpid()`.

In [3]:
# Worker-local FaceMesh registry  —  do NOT pickle this dict
_face_mesh_registry: dict = {}

def get_face_mesh() -> mp.solutions.face_mesh.FaceMesh:
    """Return a FaceMesh instance local to the current process."""
    pid = os.getpid()
    if pid not in _face_mesh_registry:
        _face_mesh_registry[pid] = mp.solutions.face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7,
        )
    return _face_mesh_registry[pid]

def worker_init_fn(worker_id: int) -> None:
    """Called by DataLoader in each worker process to pre-warm FaceMesh."""
    get_face_mesh()  # instantiate early so the first __getitem__ call is fast

In [4]:
# Landmark index groups
FOREHEAD_IDXS = [10, 67, 103, 109, 338, 297]
LEFT_CHEEK    = [50, 187, 205, 207]
RIGHT_CHEEK   = [280, 425, 411, 427]

## Skin Segmentation + ROI Extraction

**IMPROVEMENT — Skin Segmentation:** A YCrCb-based skin mask is applied *inside* each ROI
before computing the mean, suppressing non-skin pixels (hair, teeth, background leakage)
that contaminate the rPPG signal.

In [5]:
def skin_mask_ycrcb(roi_bgr: np.ndarray) -> np.ndarray:
    """
    Return a binary mask (0/255) selecting skin pixels in a BGR crop.
    Uses the standard YCrCb skin-colour range.
    """
    ycrcb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2YCrCb)
    mask  = cv2.inRange(ycrcb,
                        np.array([0,   133, 77],  dtype=np.uint8),
                        np.array([255, 173, 127], dtype=np.uint8))
    return mask


def extract_roi(
    frame_rgb: np.ndarray,
    landmarks,
    indices: list,
    apply_skin_mask: bool = True,
) -> tuple:
    """
    Returns:
        roi_resized  : np.ndarray (IMG_SIZE, IMG_SIZE, 3) float32 [0,1]
        rgb_mean     : np.ndarray (3,) — mean RGB of skin pixels inside ROI
    """
    h, w, _ = frame_rgb.shape
    pts = np.array(
        [(int(landmarks.landmark[i].x * w),
          int(landmarks.landmark[i].y * h)) for i in indices],
        dtype=np.int32,
    )

    geo_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(geo_mask, pts, 255)

    x, y, bw, bh = cv2.boundingRect(pts)
    if bw == 0 or bh == 0:
        return None, None

    roi_rgb = frame_rgb[y:y+bh, x:x+bw]
    mask_crop = geo_mask[y:y+bh, x:x+bw]

    if apply_skin_mask:
        roi_bgr = cv2.cvtColor(roi_rgb, cv2.COLOR_RGB2BGR)
        skin    = skin_mask_ycrcb(roi_bgr)
        combined = cv2.bitwise_and(mask_crop, skin)
    else:
        combined = mask_crop

    skin_pixels = roi_rgb[combined > 0]  # shape (N, 3)
    if len(skin_pixels) == 0:
        rgb_mean = roi_rgb.reshape(-1, 3).mean(axis=0).astype(np.float32)
    else:
        rgb_mean = skin_pixels.mean(axis=0).astype(np.float32)

    roi_resized = cv2.resize(roi_rgb, (cfg.IMG_SIZE, cfg.IMG_SIZE))
    roi_resized = roi_resized.astype(np.float32) / 255.0

    return roi_resized, rgb_mean

## Signal Processing

In [6]:
def POS_WANG(rgb_signals: list) -> np.ndarray:
    """
    Plain Orthogonal Subspace (POS) method — Wang et al. 2017.
    Input: list of length N, each element is np.ndarray(3,) RGB mean per frame.
    Output: 1-D rPPG signal of length N.
    """
    rgb = np.asarray(rgb_signals, dtype=np.float32)   # (N, 3)
    mean_rgb = np.mean(rgb, axis=0)                    # (3,)
    rgb = rgb / (mean_rgb + 1e-8)                      # normalise

    projection = np.array([[0, 1, -1],
                            [-2, 1,  1]], dtype=np.float32)
    S    = projection @ rgb.T                          # (2, N)
    std1 = np.std(S[0])
    std2 = np.std(S[1])
    alpha = std1 / (std2 + 1e-8)
    h = S[0] + alpha * S[1]                            # (N,)
    return h


def bandpass_filter(signal: np.ndarray, fs: float = 30.0) -> np.ndarray:
    """
    5th-order Butterworth bandpass 0.75–3.0 Hz (45–180 BPM).
    Guards against signals that are too short for filtfilt.
    """
    lowcut, highcut = 0.75, 3.0
    nyquist = fs * 0.5
    b, a = butter(5, [lowcut / nyquist, highcut / nyquist], btype='band')
    # filtfilt requires at least padlen+1 samples (≈18 for order-5)
    min_len = 3 * max(len(a), len(b))
    if len(signal) < min_len:
        return (signal - signal.mean()) / (signal.std() + 1e-8)
    filtered = filtfilt(b, a, signal)
    filtered = (filtered - filtered.mean()) / (filtered.std() + 1e-8)
    return filtered


def compute_bpm(signal: np.ndarray, fs: float = 30.0) -> float:
    signal  = bandpass_filter(signal, fs)
    freqs   = np.fft.rfftfreq(len(signal), d=1.0 / fs)
    fft_mag = np.abs(np.fft.rfft(signal))
    mask    = (freqs >= 0.75) & (freqs <= 3.0)
    bpm     = freqs[mask][np.argmax(fft_mag[mask])] * 60.0
    return float(bpm)

## Motion Augmentation

**IMPROVEMENT — Motion Augmentation:** Applied at the frame-sequence level during training.

In [7]:
def augment_sequence(frames: np.ndarray, signal: np.ndarray, training: bool = True):
    """
    frames : (T, H, W, 3) float32 [0,1]
    signal : (T,)          float32

    Augmentations (training only):
      - Random horizontal flip  (p=0.5)
      - Random brightness/contrast jitter
      - Random temporal jitter (drop/duplicate 1-3 frames from start)
    """
    if not training:
        return frames, signal

    # Horizontal flip
    if random.random() < 0.5:
        frames = frames[:, :, ::-1, :].copy()

    # Brightness / contrast
    alpha = random.uniform(0.8, 1.2)   # contrast
    beta  = random.uniform(-0.05, 0.05)  # brightness
    frames = np.clip(alpha * frames + beta, 0.0, 1.0).astype(np.float32)

    # Temporal jitter — drop 0–3 frames from the start
    jitter = random.randint(0, 3)
    if jitter > 0 and len(frames) > jitter:
        frames = frames[jitter:]
        signal = signal[jitter:]

    return frames, signal

## Temporal Difference

In [8]:
def temporal_difference(frames: np.ndarray) -> np.ndarray:
    """Frame-to-frame difference.  Output length = len(frames) - 1."""
    diff = frames[1:].astype(np.float32) - frames[:-1].astype(np.float32)
    return diff

## Video Processing

**FIX #1 — rgb_signals 3× inflation:**  
The original code appended one `rgb_mean` per ROI per frame → 3 entries per frame.
`POS_WANG` then received a signal 3× the intended frame rate, completely distorting frequency.
Fix: average the three ROI means into a single entry per frame.

**FIX #2 — signal/frame length alignment:**  
`temporal_difference` reduces frame count by 1.  The signal must be trimmed *after* this step.

**FIX #5 — empty-detection guard:**  
If fewer than `CFG.MIN_FRAMES` valid frames are found, a `ValueError` is raised instead of
silently returning empty arrays that crash downstream.

In [9]:
def process_video(video_path: str, training: bool = False):
    """
    Returns:
        diff_frames : (T-1, H, W, 3) float32   — temporal-difference frames (motion stream)
        raw_frames  : (T-1, H, W, 3) float32   — raw frames aligned to diff_frames (appearance stream)
        signal      : (T-1,)         float32   — bandpass-filtered POS rPPG signal

    FIX #9: video_path must point to a VIDEO FILE (mp4 / avi / etc.), not a directory.
    """
    if not os.path.isfile(video_path):
        raise FileNotFoundError(
            f"video_path must be a video FILE, not a directory.\n"
            f"Got: {video_path}\n"
            f"Example fix: '/data/UBFC/subject3/vid.avi'"
        )

    cap = cv2.VideoCapture(video_path)
    face_mesh = get_face_mesh()   # FIX #3: per-process instance

    raw_frames  = []
    rgb_signals = []   # ONE entry per frame (not per ROI)

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results   = face_mesh.process(frame_rgb)

        if not results.multi_face_landmarks:   # FIX #5: skip frames with no face
            continue

        landmarks = results.multi_face_landmarks[0]
        roi_imgs, roi_means = [], []

        for indices in [FOREHEAD_IDXS, LEFT_CHEEK, RIGHT_CHEEK]:
            roi_img, rgb_mean = extract_roi(frame_rgb, landmarks, indices)
            if roi_img is not None:
                roi_imgs.append(roi_img)
                roi_means.append(rgb_mean)

        if len(roi_imgs) == 0:
            continue

        # FIX #1: average across ROIs → exactly ONE rgb entry per frame
        face_img  = np.mean(np.stack(roi_imgs),  axis=0).astype(np.float32)
        frame_mean = np.mean(np.stack(roi_means), axis=0)  # (3,)

        raw_frames.append(face_img)
        rgb_signals.append(frame_mean)          # 1 entry per frame  ✓

    cap.release()

    if len(raw_frames) < cfg.MIN_FRAMES:
        raise ValueError(
            f"Only {len(raw_frames)} valid frames detected in {video_path}. "
            f"Minimum required: {cfg.MIN_FRAMES}."
        )

    raw_frames = np.array(raw_frames)  # (N, H, W, 3)

    # --- Motion stream: temporal difference ---
    diff_frames = temporal_difference(raw_frames)           # (N-1, H, W, 3)
    raw_frames  = raw_frames[:-1]                           # align to (N-1, H, W, 3)

    # --- Normalise diff frames ---
    mean = diff_frames.mean(axis=(0, 1, 2), keepdims=True)
    std  = diff_frames.std(axis=(0, 1, 2),  keepdims=True)
    diff_frames = (diff_frames - mean) / (std + 1e-8)

    # --- rPPG signal: FIX #2 — trim to same length as frames after differencing ---
    signal = POS_WANG(rgb_signals)           # length N
    signal = signal[:-1]                     # FIX #2: trim to N-1 to match diff_frames
    signal = bandpass_filter(signal, fs=cfg.FPS)

    # --- Augmentation ---
    diff_frames, signal = augment_sequence(diff_frames, signal, training=training)
    raw_frames  = raw_frames[:len(diff_frames)]

    return diff_frames, raw_frames, signal

## Dataset

**FIX #4 — Variable sequence length:**  
Each video can have a different frame count.  The original code passed raw variable-length tensors
to `DataLoader`, which crashes when the default `collate_fn` tries to stack them.  
Fix: trim to `cfg.SEQ_LEN` if long, zero-pad if short.  A validity mask is returned so the model
and loss function can ignore padding.

In [10]:
def pad_or_crop(arr: np.ndarray, length: int, axis: int = 0) -> np.ndarray:
    """Trim to `length` along `axis`, or zero-pad if shorter."""
    n = arr.shape[axis]
    if n >= length:
        return np.take(arr, range(length), axis=axis)
    pad_width = [(0, 0)] * arr.ndim
    pad_width[axis] = (0, length - n)
    return np.pad(arr, pad_width, mode='constant', constant_values=0.0)


class RPPGDataset(Dataset):

    def __init__(self, video_paths: list, training: bool = True):
        self.video_paths = video_paths
        self.training    = training

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        diff_frames, raw_frames, signal = process_video(video_path, training=self.training)

        T = len(signal)

        # FIX #4: enforce fixed SEQ_LEN for collation
        diff_frames = pad_or_crop(diff_frames, cfg.SEQ_LEN, axis=0)  # (SEQ_LEN, H, W, 3)
        raw_frames  = pad_or_crop(raw_frames,  cfg.SEQ_LEN, axis=0)  # (SEQ_LEN, H, W, 3)
        signal      = pad_or_crop(signal,      cfg.SEQ_LEN, axis=0)  # (SEQ_LEN,)

        # Validity mask: 1 for real frames, 0 for padding
        mask = np.zeros(cfg.SEQ_LEN, dtype=np.float32)
        mask[:min(T, cfg.SEQ_LEN)] = 1.0

        diff_frames = torch.tensor(diff_frames, dtype=torch.float32)
        raw_frames  = torch.tensor(raw_frames,  dtype=torch.float32)
        signal      = torch.tensor(signal,      dtype=torch.float32)
        mask        = torch.tensor(mask,         dtype=torch.float32)

        return diff_frames, raw_frames, signal, mask

## Multi-Scale CNN Encoder with Attention Pooling

**IMPROVEMENTS:**
- **Attention Pooling** replaces `AdaptiveAvgPool2d` — learns where to look spatially.
- **Multi-Scale** — features extracted at three spatial resolutions are concatenated,
  then projected to `D_MODEL`.  Inspired by TS-CAN and EfficientPhys.
- **FIX #7** — `forward()` now reshapes to `(B*T, C, H, W)` for a single batched CNN call
  instead of looping T times.

In [11]:
class AttentionPool2d(nn.Module):
    """Spatial attention pooling — learns a soft mask over spatial locations."""

    def __init__(self, in_channels: int):
        super().__init__()
        self.attn = nn.Conv2d(in_channels, 1, kernel_size=1, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        w = torch.softmax(self.attn(x).flatten(2), dim=-1)   # (B, 1, H*W)
        x = x.flatten(2)                                      # (B, C, H*W)
        return (x * w).sum(dim=-1)                            # (B, C)


class MultiScaleSpatialEncoder(nn.Module):
    """
    Dual-stream multi-scale CNN encoder.

    Processes ONE spatial frame (H, W, 3) → feature vector (D_MODEL,).
    Three parallel branches at different depths share the early stem.
    """

    def __init__(self, d_model: int = 128):
        super().__init__()

        def conv_bn_relu(ci, co, k=3, s=1, p=1):
            return nn.Sequential(
                nn.Conv2d(ci, co, k, stride=s, padding=p, bias=False),
                nn.BatchNorm2d(co),
                nn.ReLU(inplace=True),
            )

        # Shared stem
        self.stem = nn.Sequential(
            conv_bn_relu(3, 32),
            conv_bn_relu(32, 32),
            nn.MaxPool2d(2),          # H/2
        )

        # Scale 1 — shallow (after stem)
        self.scale1 = nn.Sequential(
            conv_bn_relu(32, 32),
        )
        self.pool1 = AttentionPool2d(32)

        # Scale 2 — mid
        self.scale2 = nn.Sequential(
            conv_bn_relu(32, 64),
            conv_bn_relu(64, 64),
            nn.MaxPool2d(2),          # H/4
        )
        self.pool2 = AttentionPool2d(64)

        # Scale 3 — deep
        self.scale3 = nn.Sequential(
            conv_bn_relu(64, 128),
            conv_bn_relu(128, 128),
            nn.MaxPool2d(2),          # H/8
        )
        self.pool3 = AttentionPool2d(128)

        # Fuse multi-scale features: 32 + 64 + 128 = 224
        self.fc = nn.Sequential(
            nn.Linear(32 + 64 + 128, d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.DROPOUT),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        s = self.stem(x)

        f1 = self.pool1(self.scale1(s))              # (B, 32)
        f2_feat = self.scale2(s)
        f2 = self.pool2(f2_feat)                     # (B, 64)
        f3 = self.pool3(self.scale3(f2_feat))        # (B, 128)

        fused = torch.cat([f1, f2, f3], dim=-1)      # (B, 224)
        return self.fc(fused)                        # (B, D_MODEL)

## Positional Encoding

**FIX #6:** The original transformer had **no positional encoding** — it was permutation-invariant
and could not distinguish time step 1 from time step 300.  Standard sinusoidal PE added.

In [12]:
class SinusoidalPositionalEncoding(nn.Module):

    def __init__(self, d_model: int, max_len: int = 2000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)  # (L, D)
        pos = torch.arange(max_len).unsqueeze(1).float()  # (L, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)  # (1, L, D)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## Dual-Stream rPPG Transformer

**IMPROVEMENTS:**
- **Dual stream** — separate `MultiScaleSpatialEncoder` for the motion stream (diff frames)
  and the appearance stream (raw frames), fused before the transformer.  Inspired by TS-CAN.
- **FIX #6** — Sinusoidal positional encoding added.
- **FIX #8** — `dim_feedforward=512` (was 2048).
- **FIX #7** — CNN called once with `(B*T, C, H, W)` reshape.
- **Gradient Checkpointing** — reduces activation memory for long sequences.

In [13]:
class RPPGTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # Dual-stream spatial encoders
        self.motion_enc     = MultiScaleSpatialEncoder(cfg.D_MODEL)  # motion (diff) stream
        self.appearance_enc = MultiScaleSpatialEncoder(cfg.D_MODEL)  # raw frame stream

        # Stream fusion: 2 × D_MODEL → D_MODEL
        self.stream_fuse = nn.Sequential(
            nn.Linear(cfg.D_MODEL * 2, cfg.D_MODEL),
            nn.ReLU(inplace=True),
        )

        # FIX #6: positional encoding
        self.pos_enc = SinusoidalPositionalEncoding(
            cfg.D_MODEL, max_len=cfg.SEQ_LEN + 10, dropout=cfg.DROPOUT
        )

        # Transformer — FIX #8: dim_feedforward=512
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.NHEAD,
            dim_feedforward=cfg.DIM_FF,
            dropout=cfg.DROPOUT,
            batch_first=True,
            norm_first=True,   # Pre-LN — more stable training
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=cfg.NUM_LAYERS,
            enable_nested_tensor=False,   # avoids warnings on masked input
        )

        # Local temporal refinement
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(cfg.D_MODEL, cfg.D_MODEL, 3, padding=1, bias=False),
            nn.BatchNorm1d(cfg.D_MODEL),
            nn.ReLU(inplace=True),
            nn.Conv1d(cfg.D_MODEL, cfg.D_MODEL, 3, padding=1, bias=False),
        )

        self.head = nn.Linear(cfg.D_MODEL, 1)

    def _encode_stream(self, encoder: nn.Module, x: torch.Tensor) -> torch.Tensor:
        """
        FIX #7: Batched CNN call instead of T-iteration loop.
        x  : (B, T, H, W, C)
        out: (B, T, D_MODEL)
        """
        B, T, H, W, C = x.shape
        x = x.permute(0, 1, 4, 2, 3).reshape(B * T, C, H, W)  # (B*T, C, H, W)
        if cfg.GRAD_CKPT and self.training:
            feats = checkpoint(encoder, x, use_reentrant=False)  # memory-efficient
        else:
            feats = encoder(x)
        return feats.view(B, T, -1)                             # (B, T, D_MODEL)

    def forward(
        self,
        diff_frames: torch.Tensor,
        raw_frames:  torch.Tensor,
        src_key_padding_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        diff_frames : (B, T, H, W, 3)   — motion stream
        raw_frames  : (B, T, H, W, 3)   — appearance stream
        src_key_padding_mask: (B, T) bool — True where PADDING (ignored positions)
        """
        m = self._encode_stream(self.motion_enc,     diff_frames)  # (B, T, D)
        a = self._encode_stream(self.appearance_enc, raw_frames)   # (B, T, D)

        # Fuse streams
        x = self.stream_fuse(torch.cat([m, a], dim=-1))  # (B, T, D)

        # Positional encoding
        x = self.pos_enc(x)

        # Transformer with optional padding mask
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)  # (B, T, D)

        # Temporal refinement
        x = self.temporal_conv(x.permute(0, 2, 1)).permute(0, 2, 1)  # (B, T, D)

        # Output head
        return self.head(x).squeeze(-1)  # (B, T)

## Loss Functions

**IMPROVEMENT — Enhanced Spectral Loss:**  
The original spectral loss compared full FFT magnitudes.  The improved version masks to the
physiological BPM band (0.75–3.0 Hz) and adds KL-divergence on the normalised spectrum,
encouraging the model to produce a sharp, correctly-placed spectral peak.

In [14]:
def negative_pearson(preds: torch.Tensor, labels: torch.Tensor,
                     mask: torch.Tensor = None) -> torch.Tensor:
    """
    preds, labels: (B, T)
    mask         : (B, T) — 1 = valid, 0 = padding
    """
    if mask is not None:
        # Zero-out padded positions so they don't shift the mean
        preds  = preds  * mask
        labels = labels * mask

    preds_c  = preds  - preds.mean(dim=1,  keepdim=True)
    labels_c = labels - labels.mean(dim=1, keepdim=True)

    num   = torch.sum(preds_c * labels_c, dim=1)
    denom = torch.sqrt(
        torch.sum(preds_c ** 2, dim=1) * torch.sum(labels_c ** 2, dim=1)
    )
    return (1 - num / (denom + 1e-8)).mean()


def enhanced_spectral_loss(
    pred:   torch.Tensor,
    target: torch.Tensor,
    fs:     float = 30.0,
) -> torch.Tensor:
    """
    Frequency-domain loss restricted to the physiological BPM band (0.75–3.0 Hz).
    Combines L1 magnitude difference + KL divergence on the normalised spectrum.

    pred, target: (B, T)
    """
    T = pred.shape[1]
    freqs = torch.fft.rfftfreq(T, d=1.0 / fs).to(pred.device)   # (F,)
    mask  = (freqs >= 0.75) & (freqs <= 3.0)                    # band mask

    pred_mag = torch.abs(torch.fft.rfft(pred,   dim=1))[:, mask]  # (B, F_band)
    gt_mag   = torch.abs(torch.fft.rfft(target, dim=1))[:, mask]  # (B, F_band)

    l1 = torch.mean(torch.abs(pred_mag - gt_mag))

    # KL on normalised spectra  — encourages matching the peak location
    pred_norm = pred_mag / (pred_mag.sum(dim=1, keepdim=True) + 1e-8)
    gt_norm   = gt_mag   / (gt_mag.sum(dim=1,   keepdim=True) + 1e-8)
    kl = F.kl_div(torch.log(pred_norm + 1e-8), gt_norm, reduction='batchmean')

    return l1 + 0.1 * kl

## Metrics

In [15]:
def mean_absolute_bpm_error(pred_signals: np.ndarray, gt_signals: np.ndarray,
                             fs: float = 30.0) -> float:
    """
    Compute mean |predicted BPM - ground-truth BPM| over a batch.
    pred_signals, gt_signals: (B, T) numpy arrays
    """
    errors = []
    for p, g in zip(pred_signals, gt_signals):
        bpm_p = compute_bpm(p.astype(np.float64), fs)
        bpm_g = compute_bpm(g.astype(np.float64), fs)
        errors.append(abs(bpm_p - bpm_g))
    return float(np.mean(errors))

## Training Setup

In [16]:
model = RPPGTransformer().to(cfg.DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.LR,
    weight_decay=1e-2,
    betas=(0.9, 0.95),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.EPOCHS,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 1,576,199


## Training & Validation Loops

**FIX #10:** Added validation loop with loss + MAE BPM metric.
The padding mask is passed to both the model and the Pearson loss.

In [17]:
def train_one_epoch(loader: DataLoader) -> float:
    model.train()
    total_loss = 0.0

    for diff_frames, raw_frames, signal, mask in tqdm(loader, desc="Train"):
        diff_frames = diff_frames.to(cfg.DEVICE)
        raw_frames  = raw_frames.to(cfg.DEVICE)
        signal      = signal.to(cfg.DEVICE)
        mask        = mask.to(cfg.DEVICE)       # (B, T)  1=valid, 0=pad

        # Transformer expects True = ignore/padding
        pad_mask = (mask == 0)

        optimizer.zero_grad()
        pred = model(diff_frames, raw_frames, src_key_padding_mask=pad_mask)  # (B, T)

        pearson = negative_pearson(pred, signal, mask=mask)
        smooth  = F.smooth_l1_loss(pred * mask, signal * mask)
        spec    = enhanced_spectral_loss(pred, signal)

        loss = 0.5 * pearson + 0.3 * smooth + 0.2 * spec
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def validate(loader: DataLoader) -> tuple:
    """Returns (val_loss, mean_bpm_mae)."""
    model.eval()
    total_loss = 0.0
    all_preds, all_gts = [], []

    for diff_frames, raw_frames, signal, mask in tqdm(loader, desc="Val  "):
        diff_frames = diff_frames.to(cfg.DEVICE)
        raw_frames  = raw_frames.to(cfg.DEVICE)
        signal      = signal.to(cfg.DEVICE)
        mask        = mask.to(cfg.DEVICE)
        pad_mask    = (mask == 0)

        pred = model(diff_frames, raw_frames, src_key_padding_mask=pad_mask)

        pearson = negative_pearson(pred, signal, mask=mask)
        smooth  = F.smooth_l1_loss(pred * mask, signal * mask)
        spec    = enhanced_spectral_loss(pred, signal)
        loss    = 0.5 * pearson + 0.3 * smooth + 0.2 * spec
        total_loss += loss.item()

        all_preds.append(pred.cpu().numpy())
        all_gts.append(signal.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_gts   = np.concatenate(all_gts)
    mae_bpm   = mean_absolute_bpm_error(all_preds, all_gts, fs=cfg.FPS)

    return total_loss / max(len(loader), 1), mae_bpm

## Dataset Paths

**FIX #9:** `video_paths` must point to **video files** (`.avi`, `.mp4`, etc.), not directories.
The UBFC-rPPG dataset stores each subject video as `vid.avi` inside the subject folder.

In [18]:
# ─────────────────────────────────────────────────────────────────
#  FIX #9: point to VIDEO FILES, not subject directories.
#
#  UBFC-rPPG dataset structure:
#    UBFC/dataset2/subject1/vid.avi
#    UBFC/dataset2/subject2/vid.avi
#    ...
#
#  You can auto-discover all vid.avi files like this:
#    import glob
#    all_videos = sorted(glob.glob("/data/UBFC/dataset2/*/vid.avi"))
# ─────────────────────────────────────────────────────────────────

UBFC_ROOT = "/home/dhruv/Documents/rppg project/UBFC/dataset 2"  # ← change if needed

import glob
all_video_paths = sorted(glob.glob(os.path.join(UBFC_ROOT, "*/vid.avi")))

if len(all_video_paths) == 0:
    # Fallback: try .mp4
    all_video_paths = sorted(glob.glob(os.path.join(UBFC_ROOT, "*/*.mp4")))

print(f"Found {len(all_video_paths)} video files.")
for p in all_video_paths[:5]:
    print(" ", p)

assert len(all_video_paths) > 0, (
    "No video files found.  Check UBFC_ROOT and ensure files are named vid.avi or *.mp4"
)

# Train / val split (80/20)
split = int(0.8 * len(all_video_paths))
train_paths = all_video_paths[:split]
val_paths   = all_video_paths[split:]
print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

Found 31 video files.
  /home/dhruv/Documents/rppg project/UBFC/dataset 2/subject1/vid.avi
  /home/dhruv/Documents/rppg project/UBFC/dataset 2/subject10/vid.avi
  /home/dhruv/Documents/rppg project/UBFC/dataset 2/subject11/vid.avi
  /home/dhruv/Documents/rppg project/UBFC/dataset 2/subject12/vid.avi
  /home/dhruv/Documents/rppg project/UBFC/dataset 2/subject13/vid.avi
Train: 24 | Val: 7


In [19]:
train_dataset = RPPGDataset(train_paths, training=True)
val_dataset   = RPPGDataset(val_paths,   training=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    worker_init_fn=worker_init_fn,   # FIX #3: per-worker FaceMesh init
    pin_memory=(cfg.DEVICE == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    worker_init_fn=worker_init_fn,
    pin_memory=(cfg.DEVICE == "cuda"),
)

## Start Training

In [ ]:
best_mae   = float("inf")
best_epoch = 0

for epoch in range(cfg.EPOCHS):
    train_loss = train_one_epoch(train_loader)

    if len(val_paths) > 0:
        val_loss, mae_bpm = validate(val_loader)
        print(f"Epoch {epoch+1:3d}/{cfg.EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"MAE BPM: {mae_bpm:.2f}")

        if mae_bpm < best_mae:
            best_mae   = mae_bpm
            best_epoch = epoch + 1
            torch.save(model.state_dict(), "best_rppg_transformer.pth")
    else:
        print(f"Epoch {epoch+1:3d}/{cfg.EPOCHS} | Train Loss: {train_loss:.4f}")

torch.save(model.state_dict(), "final_rppg_transformer.pth")
print(f"\nTraining complete. Best MAE {best_mae:.2f} BPM at epoch {best_epoch}.")
print("Model saved → final_rppg_transformer.pth")

Train:   0%|                                             | 0/12 [00:00<?, ?it/s]

## Self-Supervised Pre-training Hook *(Future Improvement)*

Stub for contrastive / masked-autoencoder pre-training on unlabelled face videos.  
Attach `ContrastiveRPPGHead` to a frozen `RPPGTransformer` backbone and train with
a contrastive loss (e.g. SimCLR / BYOL) before fine-tuning with the supervised pipeline above.

In [ ]:
class ContrastiveRPPGHead(nn.Module):
    """
    MLP projection head for self-supervised pre-training.
    Attach to RPPGTransformer's temporal features before the regression head.

    Usage (pseudo-code):
        backbone = RPPGTransformer()  # remove .head for feature extraction
        proj = ContrastiveRPPGHead(cfg.D_MODEL)
        z = proj(backbone_features)  # (B, proj_dim)
        loss = nt_xent_loss(z_a, z_b)  # SimCLR / BYOL objective
    """

    def __init__(self, d_model: int = 128, proj_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Linear(d_model, proj_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D) → pool over time → (B, D) → project
        return self.net(x.mean(dim=1))

print("ContrastiveRPPGHead stub ready for self-supervised pre-training.")

# Summary of All Changes

## Bugs Fixed
| # | Bug | Fix |
|---|-----|-----|
| 1 | `rgb_signals` had 3× too many entries | Average 3 ROI means → 1 per frame |
| 2 | Signal–frame length mismatch after `temporal_difference` | Trim signal by 1 after differencing |
| 3 | Global `face_mesh` not picklable → DataLoader worker crash | Per-process registry + `worker_init_fn` |
| 4 | Variable sequence lengths crash `collate_fn` | `pad_or_crop` to `SEQ_LEN` + validity mask |
| 5 | No guard for empty face detection | `ValueError` if < `MIN_FRAMES` valid frames |
| 6 | No positional encoding in Transformer | Sinusoidal PE added |
| 7 | Frame-by-frame CNN loop → slow / OOM | Reshape to `(B*T, C, H, W)`, single batched call |
| 8 | `dim_feedforward=2048` too large for `d_model=128` | Set to 512 |
| 9 | `video_paths` pointed to directory | `glob` for video files + `FileNotFoundError` |
| 10 | No validation or metrics | Val loop + MAE BPM + best-model checkpoint |

## Future Improvements Implemented
- ✅ **Attention Pooling** (`AttentionPool2d`)
- ✅ **Multi-Scale CNN** (`MultiScaleSpatialEncoder`)
- ✅ **Dual-Stream (TS-CAN inspired)** — motion + appearance branches
- ✅ **Skin Segmentation** — YCrCb mask inside `extract_roi`
- ✅ **Motion Augmentation** — flip, brightness, temporal jitter
- ✅ **Enhanced Frequency-Domain Supervision** — BPM-band L1 + KL spectral loss
- ✅ **Gradient Checkpointing** — `torch.utils.checkpoint` on CNN encoder
- ✅ **Self-Supervised Pre-training Stub** — `ContrastiveRPPGHead`

## Still Future (not yet implemented)
- PhysFormer full architecture (cross-attention between face patches)
- EfficientPhys lightweight backbone
- Full SimCLR / BYOL training loop for SSL